In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [18]:
# To read the second sheet (index 1)
private_18_19 = pd.read_excel('organized_education_data/private_schools_2018-2019.xlsx', sheet_name="Schools, Classes & Staff by Pro")
private_19_20 = pd.read_excel('organized_education_data/private_schools_2019-2020.xlsx', sheet_name="Schools, Classes & Staff by Pro")
private_20_21 = pd.read_excel('organized_education_data/private_schools_2020-2021.xlsx', sheet_name="Schools, Classes & Staff by Pro")

In [19]:
row =private_18_19.iloc[0]
private_18_19 = private_18_19.rename(columns=row)
private_18_19 = private_18_19.iloc[2:]
private_18_19 = private_18_19.reset_index(drop=True)
private_18_19 = private_18_19.set_index("Province")
cols = list(private_18_19.columns)
cols[3] = "Enrollment(F)"
cols[5] = "Teaching Staff(F)"
cols[7] = "Non-Teaching Staff(F)"
cols[10] = "Foreigner Teaching Staff(F)"
private_18_19.columns = cols
private_18_19 = private_18_19.astype('Int64')

In [20]:
private_19_20 =private_19_20.iloc[1:]
private_19_20 = private_19_20.reset_index(drop=True)
private_19_20 = private_19_20.set_index("Province")
cols = list(private_19_20.columns)
cols[3] = "Enrollment(F)"
cols[5] = "Teaching Staff(F)"
cols[7] = "Non-Teaching Staff(F)"
cols[10] = "Foreigner Teaching Staff(F)"
private_19_20.columns = cols
private_19_20 = private_19_20.astype('Int64')

In [21]:
row =private_20_21.iloc[0]
private_20_21 = private_20_21.rename(columns=row)
private_20_21 = private_20_21.iloc[2:]
private_20_21 = private_20_21.reset_index(drop=True)
private_20_21 = private_20_21.set_index("Province")
cols = list(private_20_21.columns)
cols[3] = "Enrollment(F)"
cols[5] = "Teaching Staff(F)"
cols[7] = "Non-Teaching Staff(F)"
cols[9] = "Foreigner Teaching Staff(F)"
private_20_21.columns = cols
private_20_21 = private_20_21.astype('Int64')

In [22]:
private_18_19['Year'] = '2018'
private_19_20['Year'] = '2019'
private_20_21['Year'] = '2020'

In [23]:
combined_df = pd.concat([private_18_19, private_19_20, private_20_21], ignore_index=True)

In [24]:
# 1. Get a list of all current columns
cols = list(combined_df.columns)

# 2. Pop 'Year' out of its current spot and insert it at index 0
cols.insert(0, cols.pop(cols.index('Year')))

# 3. Re-order the DataFrame
combined_df = combined_df[cols]

In [25]:
def set_province_index(df):
    """Rename the first column to Province, drop junk rows, set as index."""
    df = df.rename(columns={df.columns[0]: "Province"})
    # Drop rows where Province is NaN (header-junk rows 0 and 1)
    df = df[df["Province"].notna()].copy()
    df = df.reset_index(drop=True)
    df = df.set_index("Province")
    return df


# ═══════════════════════════════════════════════════════════════
# 1.  ENROLLMENT & REPEATERS  (Enroll_and_repeat_18_19 / 19_20 / 20_21)
# ═══════════════════════════════════════════════════════════════

Enroll_and_repeat_18_19 = pd.read_excel(
    'organized_education_data/private_schools_2018-2019.xlsx',
    sheet_name="Enrollment & Repeaters by Grade"
)
Enroll_and_repeat_19_20 = pd.read_excel(
    'organized_education_data/private_schools_2019-2020.xlsx',
    sheet_name="Enrollment & Repeaters by Grade"
)
Enroll_and_repeat_20_21 = pd.read_excel(
    'organized_education_data/private_schools_2020-2021.xlsx',
    sheet_name="Enrollment & Repeaters by Grade"
)

# Build clean column names (same for all three years)
sub_headers = ['Enrollment_Total', 'Enrollment_Girl', 'Repeaters_Total', 'Repeaters_Girl']
enroll_cols = ['Province']
for i in range(1, 13):
    for sub in sub_headers:
        enroll_cols.append(f'Grade_{i}_{sub}')
for sub in sub_headers:
    enroll_cols.append(f'Total_{sub}')


def clean_enroll(df):
    df.columns = enroll_cols
    # The junk sub-header rows (Enrollment/Total labels) have NaN in Province — drop them
    df = df[df["Province"].notna()].copy()
    df = df.reset_index(drop=True)
    df = df.set_index("Province")
    # Remove aggregate rows kept for reference (optional – comment out to keep them)
    # df = df[~df.index.isin(["Whole Kingdom", "- Urban Area", "- Rural Area"])]
    df = df.apply(pd.to_numeric, errors='coerce').astype('Int64')
    return df


Enroll_and_repeat_18_19 = clean_enroll(Enroll_and_repeat_18_19)
Enroll_and_repeat_19_20 = clean_enroll(Enroll_and_repeat_19_20)
Enroll_and_repeat_20_21 = clean_enroll(Enroll_and_repeat_20_21)

# Add Year and combine
Enroll_and_repeat_18_19['Year'] = 2018
Enroll_and_repeat_19_20['Year'] = 2019
Enroll_and_repeat_20_21['Year'] = 2020

combined_enroll = pd.concat(
    [Enroll_and_repeat_18_19, Enroll_and_repeat_19_20, Enroll_and_repeat_20_21]
)
# Move Year to front
cols = list(combined_enroll.columns)
cols.insert(0, cols.pop(cols.index('Year')))
combined_enroll = combined_enroll[cols]


# ═══════════════════════════════════════════════════════════════
# 2.  TEACHING STAFF BY EDUCATION LEVEL (Staff_18_19 / 19_20 / 20_21)
# ═══════════════════════════════════════════════════════════════

Staff_18_19 = pd.read_excel(
    'organized_education_data/private_schools_2018-2019.xlsx',
    sheet_name="Teaching Staff by Education Lev"
)
Staff_19_20 = pd.read_excel(
    'organized_education_data/private_schools_2019-2020.xlsx',
    sheet_name="Teaching Staff by Education Lev"
)
Staff_20_21 = pd.read_excel(
    'organized_education_data/private_schools_2020-2021.xlsx',
    sheet_name="Teaching Staff by Education Lev"
)

# Column structure (19 columns):
# Province | Total: Primary L.sec U.Sec Graduate PostGrad PhD
#           | Female: Primary L.sec U.Sec Graduate PostGrad PhD
#           | Without Ped Training: Primary L.sec U.Sec Graduate PostGrad PhD
staff_cols = ['Province',
    'Total_Primary', 'Total_LSec', 'Total_USec', 'Total_Graduate', 'Total_PostGrad', 'Total_PhD',
    'Female_Primary', 'Female_LSec', 'Female_USec', 'Female_Graduate', 'Female_PostGrad', 'Female_PhD',
    'NoPedTrain_Primary', 'NoPedTrain_LSec', 'NoPedTrain_USec',
    'NoPedTrain_Graduate', 'NoPedTrain_PostGrad', 'NoPedTrain_PhD'
]


def clean_staff(df):
    df.columns = staff_cols
    # Row 0 is the sub-header labels; drop it
    df = df.iloc[1:].reset_index(drop=True)
    df = df[df["Province"].notna()].copy()
    df = df.set_index("Province")
    df = df.apply(pd.to_numeric, errors='coerce').astype('Int64')
    return df


Staff_18_19 = clean_staff(Staff_18_19)
Staff_19_20 = clean_staff(Staff_19_20)
Staff_20_21 = clean_staff(Staff_20_21)

# Add Year and combine
Staff_18_19['Year'] = 2018
Staff_19_20['Year'] = 2019
Staff_20_21['Year'] = 2020

combined_staff = pd.concat([Staff_18_19, Staff_19_20, Staff_20_21])
cols = list(combined_staff.columns)
cols.insert(0, cols.pop(cols.index('Year')))
combined_staff = combined_staff[cols]


# ═══════════════════════════════════════════════════════════════
# 3.  LEARNING FACILITIES / SCHOOL EQUIPMENT (School_18_19 / 19_20 / 20_21)
# ═══════════════════════════════════════════════════════════════

School_18_19 = pd.read_excel(
    'organized_education_data/private_schools_2018-2019.xlsx',
    sheet_name="Learning Facilities (Computers,"
)
School_19_20 = pd.read_excel(
    'organized_education_data/private_schools_2019-2020.xlsx',
    sheet_name="Learning Facilities (Computers,"
)
School_20_21 = pd.read_excel(
    'organized_education_data/private_schools_2020-2021.xlsx',
    sheet_name="Learning Facilities (Computers,"
)

# 2018-19 columns: Computer Printer LCD_Projector Photocopy Library Home_Economic Resource_Center Workshop Drinking_Water Toilet
# 2019-20 & 2020-21 columns differ slightly: Health_room instead of Home_Economic, Computer_room instead of Workshop
school_cols_18_19 = ['Province',
    'Computer', 'Printer', 'LCD_Projector', 'Photocopy', 'Library',
    'Home_Economic', 'Resource_Center', 'Workshop', 'Drinking_Water', 'Toilet'
]
school_cols_later = ['Province',
    'Computer', 'Printer', 'LCD_Projector', 'Photocopy', 'Library',
    'Health_Room', 'Resource_Center', 'Computer_Room', 'Drinking_Water', 'Toilet'
]


def clean_school(df, cols):
    df.columns = cols
    # Row 0 is the facility-name sub-header; drop it
    df = df.iloc[1:].reset_index(drop=True)
    df = df[df["Province"].notna()].copy()
    df = df.set_index("Province")
    df = df.apply(pd.to_numeric, errors='coerce').astype('Int64')
    return df


School_18_19 = clean_school(School_18_19, school_cols_18_19)
School_19_20 = clean_school(School_19_20, school_cols_later)
School_20_21 = clean_school(School_20_21, school_cols_later)

# Add Year and combine (union of all columns; missing cols filled with NA)
School_18_19['Year'] = 2018
School_19_20['Year'] = 2019
School_20_21['Year'] = 2020

combined_school = pd.concat([School_18_19, School_19_20, School_20_21])
cols = list(combined_school.columns)
cols.insert(0, cols.pop(cols.index('Year')))
combined_school = combined_school[cols]

In [26]:
combined_df

,Year,Number of\nSchools,Number of\nClasses,Enrollment,Enrollment(F),Teaching Staff,Teaching Staff(F),Non-Teaching Staff,Non-Teaching Staff(F),Govern. staff,Foreigner Teaching Staff,Foreigner Teaching Staff(F),Buildings,Rooms,Classrooms
0,2018,121,894,21942,10611,1053,580,252,113,269,49,25,132,1382,703
1,2018,45,406,8804,4371,779,488,165,82,552,38,27,57,632,361
2,2018,25,158,4010,1969,214,130,34,14,54,7,6,52,295,140
3,2018,25,190,4505,2055,356,242,87,46,132,13,9,51,326,183
4,2018,50,328,8134,4179,444,279,109,41,116,49,36,67,517,296
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,2020,52,274,5697,2797,468,241,181,91,<NA>,22,12,170,593,268
80,2020,27,164,4692,2671,243,155,34,19,<NA>,28,18,43,401,164
81,2020,1666,11895,240550,118049,15302,9179,6880,4167,<NA>,3298,1917,2663,29571,10573
82,2020,346,2096,43843,21756,2671,1524,684,323,<NA>,191,116,730,3902,1774


In [27]:
combined_enroll

,Year,Grade_1_Enrollment_Total,Grade_1_Enrollment_Girl,Grade_1_Repeaters_Total,Grade_1_Repeaters_Girl,Grade_2_Enrollment_Total,Grade_2_Enrollment_Girl,Grade_2_Repeaters_Total,Grade_2_Repeaters_Girl,Grade_3_Enrollment_Total,...,Grade_11_Repeaters_Total,Grade_11_Repeaters_Girl,Grade_12_Enrollment_Total,Grade_12_Enrollment_Girl,Grade_12_Repeaters_Total,Grade_12_Repeaters_Girl,Total_Enrollment_Total,Total_Enrollment_Girl,Total_Repeaters_Total,Total_Repeaters_Girl
Province,,,,,,,,,,,,,,,,,,,,,
Banteay Meanchey,2018,3270,1518,157,64,2972,1444,85,24,2594,...,1,0,358,167,7,1,21942,10611,392,133
Battambang,2018,1028,487,3,0,828,386,8,0,689,...,2,1,453,245,17,7,8804,4371,98,23
Kampong Cham,2018,545,278,32,10,528,253,27,13,455,...,0,0,58,28,0,0,4010,1969,93,37
Kampong Chhnang,2018,499,261,3,1,426,191,3,1,246,...,0,0,176,73,1,1,4505,2055,15,4
Kampong Speu,2018,1091,552,29,12,825,412,21,7,741,...,0,0,38,19,0,0,8134,4179,101,47
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Takeo,2020,826,384,3,1,658,328,12,6,427,...,2,1,167,88,4,2,5697,2797,42,17
Tbaung Khmum,2020,400,208,14,5,349,180,17,10,324,...,0,0,134,94,0,0,4692,2671,133,85
Whole Kingdom,2020,30941,15049,895,413,24062,12004,486,219,21534,...,20,8,8058,4194,79,37,240550,118049,2832,1331


In [28]:
combined_staff

,Year,Total_Primary,Total_LSec,Total_USec,Total_Graduate,Total_PostGrad,Total_PhD,Female_Primary,Female_LSec,Female_USec,Female_Graduate,Female_PostGrad,Female_PhD,NoPedTrain_Primary,NoPedTrain_LSec,NoPedTrain_USec,NoPedTrain_Graduate,NoPedTrain_PostGrad,NoPedTrain_PhD
Province,,,,,,,,,,,,,,,,,,,
Banteay Meanchey,2018,0,38,430,554,31,0,0,29,265,280,6,0,0,8,50,51,1,0
Battambang,2018,12,31,228,467,40,1,12,19,169,274,14,0,0,1,5,8,0,0
Kampong Cham,2018,5,23,74,103,9,0,1,17,59,53,0,0,0,0,3,8,0,0
Kampong Chhnang,2018,0,17,195,140,3,1,0,12,147,82,1,0,0,0,9,11,0,0
Kampong Speu,2018,4,20,130,271,19,0,2,16,93,159,9,0,0,0,2,13,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Takeo,2020,21,39,204,177,25,2,18,22,115,82,4,0,0,6,32,6,0,0
Tbaung Khmum,2020,1,3,96,140,3,0,1,2,69,83,0,0,0,0,7,32,1,0
Whole Kingdom,2020,582,892,4169,8568,998,93,403,638,2986,4762,359,31,75,97,485,898,77,3


In [29]:
combined_school

,Year,Computer,Printer,LCD_Projector,Photocopy,Library,Home_Economic,Resource_Center,Workshop,Drinking_Water,Toilet,Health_Room,Computer_Room
Province,,,,,,,,,,,,,
Banteay Meanchey,2018,88,82,52,67,51,24,25,14,87,107,<NA>,<NA>
Battambang,2018,36,32,28,27,36,13,16,4,42,44,<NA>,<NA>
Kampong Cham,2018,15,11,4,9,8,0,6,2,17,22,<NA>,<NA>
Kampong Chhnang,2018,19,17,15,15,11,4,4,4,24,24,<NA>,<NA>
Kampong Speu,2018,39,28,22,29,30,12,2,8,45,50,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...
Takeo,2020,706,38,49,52,37,<NA>,10,<NA>,49,46,25,41
Tbaung Khmum,2020,254,26,17,25,12,<NA>,6,<NA>,23,23,14,19
Whole Kingdom,2020,24041,1835,2977,1707,3129,<NA>,561,<NA>,1418,1225,1126,1140
